# Card win rates

R377's figures for every card, over the game records the loader reads (SPEC §9.11, R376–R378): the win rate
of the decks that held a card, of those that opened with it, of those that went first or second, and of those
that played it against those that drew it and did not, each beside its number of games. Then the cards by set
and rarity, one patch against another, and a development run against live games of the same patch.

The arithmetic is the Rust's (`jackioh_engine::card_stats`); `analysis/tests/test_card_stats.py` holds the
loader to it, so this notebook only reads and draws. A card with fewer games than the server's
`CARD_STATS_MIN_SAMPLE` is marked `*` and never ranked, as on the statistics page (R654).

**Inputs** come from the environment, and default to the committed fixtures, so a run with nothing set draws
from 40 development games and 40 more filed as live ones (`analysis/README.md`):

| Variable | What | Default |
|---|---|---|
| `JACKIOH_RECORDS` | record files, comma-separated | `analysis/fixtures/dev-40.jsonl`, `live-40.jsonl` |
| `JACKIOH_CATALOG` | the catalog | `crates/cards/catalog.json` |
| `JACKIOH_SOURCE` | `live` (the default), `dev` or `all`: the games read, as a query of R378's does | `live` |
| `JACKIOH_PATCH` | the patch to read | the newest patch in the records |
| `JACKIOH_MIN_SAMPLE` | games a card needs to be ranked | the server's `CARD_STATS_MIN_SAMPLE` |

In [ ]:
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter


def find_analysis():
    """The `analysis/` directory, whether Jupyter started here, at the repository root or anywhere below it."""
    for here in (Path.cwd(), *Path.cwd().parents):
        for candidate in (here, here / "analysis"):
            if (candidate / "jackioh_records" / "__init__.py").is_file():
                return candidate
    raise RuntimeError("Open this notebook from inside the JackiOh repository")


ANALYSIS = find_analysis()
sys.path.insert(0, str(ANALYSIS))
FIXTURES = ANALYSIS / "fixtures"


def from_repo(path):
    """A path as given: a relative one is read from the repository root, wherever Jupyter started."""
    path = Path(path).expanduser()
    return path if path.is_absolute() else ANALYSIS.parent / path


def paths_from(name, default):
    """A comma-separated list of files from the environment, or the default."""
    raw = os.environ.get(name)
    return [from_repo(part.strip()) for part in raw.split(",") if part.strip()] if raw else default


from jackioh_records import BREAKDOWNS, CATALOG_PATH, card_stats, load, load_catalog

RECORDS = paths_from("JACKIOH_RECORDS", [FIXTURES / "dev-40.jsonl", FIXTURES / "live-40.jsonl"])
CATALOG = from_repo(os.environ.get("JACKIOH_CATALOG", CATALOG_PATH))


def server_min_sample():
    """The server's `CARD_STATS_MIN_SAMPLE`, read off its config so this notebook states no number of its own."""
    config = ANALYSIS.parent / "crates" / "server" / "src" / "config.rs"
    found = re.search(r"CARD_STATS_MIN_SAMPLE: i64 = (\d+);", config.read_text(encoding="utf-8"))
    if found is None:
        raise RuntimeError(f"No CARD_STATS_MIN_SAMPLE in {config}: set JACKIOH_MIN_SAMPLE")
    return int(found[1])


MIN_SAMPLE = int(os.environ.get("JACKIOH_MIN_SAMPLE") or server_min_sample())

print("records:", *RECORDS, sep="\n  ")
print("catalog:", CATALOG)
print("a card needs", MIN_SAMPLE, "games in deck to be ranked")
if any(FIXTURES in path.resolve().parents for path in RECORDS):
    print("These are the committed fixtures, not live data: set JACKIOH_RECORDS (analysis/README.md).")

In [ ]:
plt.rcParams.update(
    {
        "figure.dpi": 100,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#8a8a85",
        "axes.labelcolor": "#52514e",
        "axes.titlecolor": "#0b0b0b",
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": "#e3e2dc",
        "grid.linewidth": 0.6,
        "legend.frameon": False,
    }
)
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK = "#0b0b0b"
MUTED = "#52514e"


def hbars(ax, labels, values, notes, color, xlabel, title, percent=False):
    """Horizontal bars, first row on top, each with its note (the games behind it) printed at its end."""
    rows = range(len(labels))
    ax.barh(rows, values, color=color, height=0.6)
    ax.set_yticks(rows, labels)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.axvline(0, color="#8a8a85", linewidth=0.8)
    for row, (value, note) in enumerate(zip(values, notes, strict=True)):
        ax.annotate(
            note,
            (value, row),
            xytext=(4 if value >= 0 else -4, 0),
            textcoords="offset points",
            va="center",
            ha="left" if value >= 0 else "right",
            fontsize=8,
            color=INK,
        )
    low, high = ax.get_xlim()
    span = high - low
    ax.set_xlim(low - 0.28 * span if low < 0 else low, high + 0.28 * span)
    if percent:
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlabel(xlabel)
    ax.set_title(title)

## The records

One line per patch, source and mode. `live` is games players played; `dev` is the AI's development runs
(R378).

In [ ]:
records = load(RECORDS)
catalog = load_catalog(CATALOG)
games = records.games
if games.empty:
    raise ValueError("No game records were read: set JACKIOH_RECORDS")


def version_key(patch):
    """A patch's order: its numbers, then its revision letter (v0.2.10 before v0.2.10b before v0.2.10c)."""
    return tuple(int(part) for part in re.findall(r"\d+", patch)), patch.lstrip("v").lstrip("0123456789.")


# `live` (the default) reads the games players played, `dev` the AI's development runs and `all` both (R378).
SOURCE = os.environ.get("JACKIOH_SOURCE") or "live"
if SOURCE not in ("live", "dev", "all"):
    raise ValueError(f"JACKIOH_SOURCE is live, dev or all, not {SOURCE!r}")
read = games if SOURCE == "all" else games[games["source"] == SOURCE]
if read.empty:
    raise ValueError(f"No {SOURCE} game in the records: set JACKIOH_SOURCE to dev or all, or JACKIOH_RECORDS")

PATCHES = sorted(read["patch"].unique(), key=version_key)
PATCH = os.environ.get("JACKIOH_PATCH") or PATCHES[-1]
if PATCH not in PATCHES:
    raise ValueError(f"No game of patch {PATCH} in the records (they hold {', '.join(PATCHES)})")
PREVIOUS = PATCHES[PATCHES.index(PATCH) - 1] if PATCHES.index(PATCH) > 0 else None

print(f"{len(games)} games; reading {SOURCE} games of patch {PATCH} (the one before it: {PREVIOUS})")
games.groupby(["patch", "source", "mode"]).size().rename("games").to_frame()

## Every card on one patch

`stats` is the whole table: R377's six breakdowns, each with `_games`, `_wins`, `_draws` and `_rate`, and
`played_delta` in percentage points. Below, the cards with the most games, each rate beside its games.

In [ ]:
def with_names(stats):
    named = stats.merge(catalog[["name", "set", "rarity", "cost"]], left_on="card", right_index=True, how="left")
    named["name"] = named["name"].fillna(named["card"])
    # A card the catalog does not hold (renamed or removed since the records were written) is still a card.
    named[["set", "rarity"]] = named[["set", "rarity"]].fillna("unknown")
    named["enough"] = named["in_deck_games"] >= MIN_SAMPLE
    return named


def stats_for(**filters):
    """R377's figures for the games the filters read, with each card's name, set, rarity and cost."""
    raw = card_stats(records, **filters)
    table = with_names(raw)
    table.attrs.update(raw.attrs)
    return table


stats = stats_for(source=SOURCE, patch=PATCH)
print(
    f"patch {PATCH}, {SOURCE} games: {stats.attrs['games']} games, {stats.attrs['decks']} decks, "
    f"{len(stats)} cards, {int(stats['enough'].sum())} of them with at least {MIN_SAMPLE} games"
)

LABELS = {
    "in_deck": "in deck",
    "opening_hand": "opening hand",
    "going_first": "going first",
    "going_second": "going second",
    "played": "played",
    "drawn_not_played": "drawn, not played",
}


def readable(stats):
    """One string per breakdown, "62% of 24": the rate beside its games ("-" with none)."""
    table = pd.DataFrame(
        {
            "card": stats["card"],
            "name": stats["name"] + stats["enough"].map({True: "", False: " *"}),
            "set": stats["set"],
            "rarity": stats["rarity"],
        }
    )
    for breakdown in BREAKDOWNS:
        games_of = stats[f"{breakdown}_games"]
        rate = stats[f"{breakdown}_rate"]
        table[LABELS[breakdown]] = ["-" if n == 0 else f"{r:.0%} of {n}" for r, n in zip(rate, games_of, strict=True)]
    table["played delta (points)"] = ["-" if pd.isna(delta) else f"{delta:+.1f}" for delta in stats["played_delta"]]
    return table.set_index("card")


print("* fewer than", MIN_SAMPLE, "games in deck: shown, never ranked")
readable(stats.sort_values(["in_deck_games", "card"], ascending=[False, True])).head(40)

## By set and rarity

Every card's games added together by set, and by rarity: the win rate of the decks that held a card of that
set or rarity, counting each card in each deck once, so a deck with three Rares adds three games to Rare. Each title
gives the rate of all cards together to read the bars against.

In [ ]:
def grouped(stats, by):
    """Cards added together: wins and games summed per group, then R377's rates over the sums."""
    columns = [f"{b}_{part}" for b in BREAKDOWNS for part in ("games", "wins")]
    sums = stats.groupby(by)[columns].sum()
    table = pd.DataFrame(index=sums.index)
    for breakdown in BREAKDOWNS:
        table[f"{breakdown}_games"] = sums[f"{breakdown}_games"]
        table[f"{breakdown}_rate"] = sums[f"{breakdown}_wins"] / sums[f"{breakdown}_games"].where(
            sums[f"{breakdown}_games"] > 0
        )
    table["played_delta"] = 100 * (table["played_rate"] - table["drawn_not_played_rate"])
    return table


overall = stats["in_deck_wins"].sum() / max(stats["in_deck_games"].sum(), 1)
RARITIES = ["Common", "Rare", "Epic", "Legendary", "Mythic"]
by_set = grouped(stats, "set")
present = set(stats["rarity"])
by_rarity = grouped(stats, "rarity").reindex([*(r for r in RARITIES if r in present), *sorted(present - set(RARITIES))])

fig, axes = plt.subplots(
    2,
    1,
    figsize=(8, 1.2 + 0.5 * (len(by_set) + len(by_rarity))),
    height_ratios=[max(len(by_set), 1), max(len(by_rarity), 1)],
)
for ax, table, title in (
    (axes[0], by_set, "In-deck win rate by set"),
    (axes[1], by_rarity, "In-deck win rate by rarity"),
):
    hbars(
        ax,
        list(table.index),
        list(table["in_deck_rate"].fillna(0)),
        [f"{rate:.1%} · {n} games" for rate, n in zip(table["in_deck_rate"], table["in_deck_games"], strict=True)],
        BLUE,
        "win rate of the decks holding a card of it (a draw is not a win)",
        f"{title}; all cards together {overall:.1%}",
        percent=True,
    )
fig.tight_layout()
plt.show()

pd.concat({"set": by_set, "rarity": by_rarity}, names=["by"]).round(3)

## The best and the worst

The 15 cards with the highest and the 15 with the lowest in-deck win rate, then the same for the played delta
(the win rate when the card was played minus the win rate when it was drawn and not played, in percentage
points). Only cards with at least the minimum games in deck are ranked; with fewer than 30 such cards the
list is split in half. The number of games is printed on each bar.

In [ ]:
def ranked_halves(stats, column, top=15):
    """The highest and the lowest `top` cards by `column` among those with enough games, never the same card twice."""
    pool = stats[stats["enough"] & stats[column].notna()].sort_values([column, "card"], ascending=[False, True])
    half = min(top, len(pool) // 2)
    return pool.head(half), pool.tail(half).iloc[::-1]


def plot_halves(stats, column, title, xlabel, note_of, percent=False):
    highest, lowest = ranked_halves(stats, column)
    if highest.empty:
        print(f"{title}: no card has {MIN_SAMPLE} games in deck yet (or too few to split), so none is ranked")
        return
    fig, axes = plt.subplots(2, 1, figsize=(9, 1.5 + 0.36 * 2 * len(highest)))
    for ax, part, color, label in ((axes[0], highest, BLUE, "highest"), (axes[1], lowest, ORANGE, "lowest")):
        hbars(
            ax,
            list(part["name"]),
            list(part[column]),
            [note_of(row) for row in part.itertuples()],
            color,
            xlabel,
            f"{title}: the {len(part)} {label}",
            percent=percent,
        )
    fig.tight_layout()
    plt.show()


plot_halves(
    stats,
    "in_deck_rate",
    "In-deck win rate",
    "win rate of the decks holding the card",
    lambda row: f"{row.in_deck_rate:.0%} · {row.in_deck_games} games",
    percent=True,
)
plot_halves(
    stats,
    "played_delta",
    "Played delta",
    "percentage points: played minus drawn and not played",
    lambda row: f"{row.played_delta:+.1f} · {row.played_games} played, {row.drawn_not_played_games} not",
)

## One patch against another

The patch before the one read, against it, for every card with enough games in both: its in-deck win rate on
each, and the change in percentage points. A card a patch changed is expected to move; a card it did not
change is mostly noise, which the games beside each rate make visible.

In [ ]:
def compare(before, after):
    """Cards with enough games in both tables: each side's in-deck rate and games, and the change in points."""
    both = before.loc[before["enough"], ["card", "name", "in_deck_rate", "in_deck_games"]].merge(
        after.loc[after["enough"], ["card", "in_deck_rate", "in_deck_games"]],
        on="card",
        suffixes=("_before", "_after"),
    )
    both["change"] = 100 * (both["in_deck_rate_after"] - both["in_deck_rate_before"])
    return both.sort_values(["change", "card"], ascending=[False, True])


def readable_change(both, before_label, after_label, rows=30):
    """The compared cards, biggest rise first, each rate beside its games; the whole table is `both`."""
    return pd.DataFrame(
        {
            "name": both["name"],
            before_label: [
                f"{r:.0%} of {n}"
                for r, n in zip(both["in_deck_rate_before"], both["in_deck_games_before"], strict=True)
            ],
            after_label: [
                f"{r:.0%} of {n}" for r, n in zip(both["in_deck_rate_after"], both["in_deck_games_after"], strict=True)
            ],
            "change (points)": both["change"].map("{:+.1f}".format),
        }
    ).head(rows)


def plot_change(both, title, before_label, after_label, top=10):
    movers = pd.concat([both.head(top), both.tail(top)]).drop_duplicates("card") if len(both) > top else both
    fig, ax = plt.subplots(figsize=(9, 1.2 + 0.32 * len(movers)))
    hbars(
        ax,
        list(movers["name"]),
        list(movers["change"]),
        [
            f"{row.in_deck_rate_before:.0%} ({row.in_deck_games_before}) → {row.in_deck_rate_after:.0%} ({row.in_deck_games_after})"
            for row in movers.itertuples()
        ],
        BLUE,
        f"change in in-deck win rate, percentage points ({before_label} → {after_label}; games in brackets)",
        title,
    )
    fig.tight_layout()
    plt.show()


if PREVIOUS is None:
    print(f"The records hold no patch before {PATCH}, so there is nothing to compare it with")
else:
    before = stats_for(source=SOURCE, patch=PREVIOUS)
    changes = compare(before, stats)
    print(
        f"{PREVIOUS}: {before.attrs['games']} games; {PATCH}: {stats.attrs['games']} games; "
        f"{len(changes)} cards have at least {MIN_SAMPLE} games on both"
    )
    if changes.empty:
        print("No card is compared yet")
    else:
        plot_change(changes, f"In-deck win rate, {PREVIOUS} → {PATCH}", PREVIOUS, PATCH)
readable_change(changes, PREVIOUS, PATCH) if PREVIOUS is not None and not changes.empty else None

## A development run against live games of the same patch

R378: until a patch has enough live games, the AI's development games stand in for them. Here the two are read
apart, on the patch above, for every card with enough games in both. A card far from the line is one the AI
plays differently from people, or one with too few games to say.

In [ ]:
dev = stats_for(source="dev", patch=PATCH)
live = stats_for(source="live", patch=PATCH)
print(f"patch {PATCH}: {dev.attrs['games']} development games, {live.attrs['games']} live games")
if dev.attrs["games"] == 0 or live.attrs["games"] == 0:
    print("Both are needed to compare them: read a development run and live games of one patch together")
    agreement = None
else:
    agreement = compare(dev, live)
    print(f"{len(agreement)} cards have at least {MIN_SAMPLE} games in both")
    if agreement.empty:
        print("No card is compared yet")
    else:
        # compare() names its sides before and after: here the development games and the live ones.
        # Spearman's rank correlation: the correlation of the ranks (scipy is not a dependency).
        rho = agreement["in_deck_rate_before"].rank().corr(agreement["in_deck_rate_after"].rank())
        print(f"rank correlation of the two in-deck win rates over those cards: {rho:.2f}")
        plot_change(agreement, f"Live against development, {PATCH}", "development", "live")
readable_change(agreement, "development", "live") if agreement is not None and not agreement.empty else None